# Study 2: When do the agents develop or mitigate?

- This notebook uses the five-seed policies saved by notebook 03.
- It looks at action choices over time on the same 20 held-out starts.

## Settings

In [24]:
import json

import pandas as pd

from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment


In [25]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
Q_TABLE_DIRECTORY = "../results/reward_comparison"
ACTION_NAMES = {0: "save", 1: "develop", 2: "low", 3: "medium", 4: "high"}

## Load the saved policies

- Each JSON file holds four Q-tables, one for each region.
- We will choose greedy actions during evaluation, with no random exploration.

In [20]:
agents_by_run = {}

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        filename = f"{Q_TABLE_DIRECTORY}/q_{reward_mode}_{training_seed}.json"
        with open(filename, encoding="utf-8") as file:
            tables = json.load(file)

        agents = []
        for region, table in enumerate(tables):
            agent = TabularQLearner(region)
            for key, values in table.items():
                observation = tuple(int(value) for value in key.split(","))
                agent.q[observation] = values
            agents.append(agent)

        agents_by_run[(reward_mode, training_seed)] = agents

print(f"Loaded {len(agents_by_run)} trained policies")

Loaded 15 trained policies


## Record action choices

- One row represents one region at one step.
- The environment still updates all four regions together.

In [21]:
def collect_episode_actions(agents, reward_mode, seed):
    environment = EconomicSustainabilityEnvironment(reward_mode)
    observations = environment.reset(seed)
    rows = []

    while True:
        actions = []
        for region in range(4):
            action = agents[region].select_action(observations[region], greedy=True)
            actions.append(action)
            rows.append({"step": environment.step_count + 1,
                         "region": region, "action": action})

        observations, rewards, terminated, truncated, info = environment.step(tuple(actions))
        if terminated or truncated:
            break

    return rows

In [22]:
action_rows = []

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = agents_by_run[(reward_mode, training_seed)]

        for environment_seed in EVALUATION_SEEDS:
            rows = collect_episode_actions(agents, reward_mode, environment_seed)
            for row in rows:
                row["reward_mode"] = reward_mode
            action_rows.extend(rows)

actions = pd.DataFrame(action_rows)
print(f"Recorded {len(actions):,} region-steps")

Recorded 240,000 region-steps


## Actions over time

- Each row of the table is one reward mode during one stage.
- The five percentages in a row add to 100.

In [23]:
actions["stage"] = pd.cut(
    actions["step"],
    bins=(0, 50, 100, 150, 200),
    labels=("1-50", "51-100", "101-150", "151-200"),
)

stage_table = pd.crosstab(
    [actions["reward_mode"], actions["stage"]],
    actions["action"],
    normalize="index",
)
display((stage_table.rename(columns=ACTION_NAMES) * 100).round(1))

action               save  develop   low  medium  high
reward_mode stage                                     
global      1-50      6.4     16.8  15.6    27.8  33.3
            51-100    4.2      1.1  10.1    18.2  66.4
            101-150   4.0      0.0  10.0    21.4  64.6
            151-200   4.0      0.0  10.0    21.5  64.5
local       1-50      3.4     21.5   1.8    18.9  54.4
            51-100    0.0      0.6   0.0     1.8  97.6
            101-150   0.0      0.0   0.0     4.9  95.1
            151-200   0.0      0.0   0.0     5.0  95.0
mixed       1-50      3.7     19.4   3.6    20.8  52.5
            51-100    0.8      0.3   0.0     2.6  96.2
            101-150   0.9      0.0   0.0     4.7  94.5
            151-200   0.9      0.0   0.0     5.0  94.1